# Bank Marketing (Fair DP-GANs) 

Author: Ilse Harmers \
Last updated: April 21, 2026

In [ ]:
# Importing libraries.
import numpy as np
import pandas as pd
from snsynth import Synthesizer
import snsynth.transform as tf
import utils
import warnings
warnings.filterwarnings("ignore", message = r"Using", category = FutureWarning)
warnings.filterwarnings("ignore", message = r"invalid", category = RuntimeWarning)
warnings.filterwarnings("ignore", message = r"divide", category = RuntimeWarning)

In [ ]:
# Importing train set.
bank_train = pd.read_csv("./train-test-datasets/bank-marketing/bank_train.csv")

# Preprocessing the dataset for the Fair DP-GANs; the dataset Fair DP-GANs are expecting the first two columns in the original dataset to be the 
# sensitive and target attributes. 
cols = bank_train.columns.to_list()
cols.remove("age")
cols.remove("y")
bank_train = bank_train[["age", "y"] + cols]

print(bank_train.columns.to_list())
bank_train.describe()

In [ ]:
# Setting up preprocessor table transformer.

tt = tf.TableTransformer([
    tf.MinMaxTransformer(lower = bank_train["age"].min(), upper = bank_train["age"].max(),
                         negative = False), # age; scaling to range (0, 1)
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # y
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # job
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # marital
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # education
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # default
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(-8000) + 1)), upper = np.log(102000 + 1), 
                             negative = False) # balance; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # housing
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # loan
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # contact
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # day
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # month
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(4900 + 1), 
                             negative = False) # duration; scaling to range (0, 1)
    ]),
    tf.MinMaxTransformer(lower = bank_train["campaign"].min(), upper = 60,
                         negative = False), # campaign; scaling to range (0, 1)
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(bank_train["pdays"].min()) + 1)), upper = np.log(870 + 1),
                             negative = False) # pdays; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(280 + 1),
                             negative = False) # previous; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # poutcome
])

In [ ]:
# Defining delta as the inverse of the size of the dataset: if a dataset has 4 * 10^4 rows, then delta = 10^(-5). 
delta = 10**np.floor(np.log10(1/bank_train.shape[0]))
print(delta)

# Defining beta1 in Adam optimizer.
beta1 = 0.9 

# Defining epsilon value.
epsi = 2

# Defining Fair DP-GAN: [dem, dis, dem-dis, clf_dem, clf_dis, clf_eqopp, clf_all].
fair_loss = "clf_all"

In [ ]:
r = 1
while r < 6:
    
    print(f"Run {r}")

    # Synthesizing the dataset with one of the Fair DP-GANs.
    synth = Synthesizer.create('fairdpgan', s = "age", s_unpriv = 25, s_priv = 25, y = "y", y_des = "yes", fair_loss = fair_loss, epochs = 10000,
                               epsilon = epsi, delta = delta, beta1 = beta1, epoch_time = True, batch_size = 512, dataset = "Bank",
                               #sanity_check = True
                              )
    synth.fit(bank_train, transformer = tt, preprocessor_eps = 0.0)

    r += 1

In [ ]:
# Saving the results on runtime.
#time_per_run = pd.DataFrame({"time": [1.026298, 1.024237, 1.025382, 1.024024, 1.015664]}, index = [f"run{i}" for i in range(1, 6)])
#time_per_run = pd.DataFrame({"time": [1.037981, 1.024933, 1.072782, 1.041885, 1.022006]}, index = [f"run{i}" for i in range(1, 6)])
#time_per_run = pd.DataFrame({"time": [1.068977, 1.053316, 1.039654, 1.033100, 1.029546]}, index = [f"run{i}" for i in range(1, 6)])
#time_per_run = pd.DataFrame({"time": [8.345443, 8.312652, 8.314166, 8.326968, 8.330530]}, index = [f"run{i}" for i in range(1, 6)])
#time_per_run = pd.DataFrame({"time": [8.315479, 8.305117, 8.298567, 8.276971, 8.290177]}, index = [f"run{i}" for i in range(1, 6)])
#time_per_run = pd.DataFrame({"time": [8.467580, 8.422815, 8.408769, 8.405629, 8.400258]}, index = [f"run{i}" for i in range(1, 6)])
#time_per_run = pd.DataFrame({"time": [8.553978, 8.553916, 8.563820, 8.600279, 8.623803]}, index = [f"run{i}" for i in range(1, 6)])
time_per_run.to_csv(f"./Runtimes/bank_runtime_{fair_loss}.csv")
time_per_run

In [ ]:
# Computing average runtime across the runs.
print("Mean runtime [s/epoch]:", np.mean(time_per_run["time"]))
print("Standard deviation runtime [s/epoch]:", np.std(time_per_run["time"]))